In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline

In [2]:
df = pd.read_csv("../files/gender_classification.csv")

In [3]:
df.shape

(5001, 8)

In [4]:
df.head()

,long_hair,forehead_width_cm,forehead_height_cm,nose_wide,nose_long,lips_thin,distance_nose_to_lip_long,gender
0,1,11.8,6.1,1,0,1,1,Male
1,0,14.0,5.4,0,0,1,0,Female
2,0,11.8,6.3,1,1,1,1,Male
3,0,14.4,6.1,0,1,1,1,Male
4,1,13.5,5.9,0,0,0,0,Female


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5001 entries, 0 to 5000
Data columns (total 8 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   long_hair                  5001 non-null   int64  
 1   forehead_width_cm          5001 non-null   float64
 2   forehead_height_cm         5001 non-null   float64
 3   nose_wide                  5001 non-null   int64  
 4   nose_long                  5001 non-null   int64  
 5   lips_thin                  5001 non-null   int64  
 6   distance_nose_to_lip_long  5001 non-null   int64  
 7   gender                     5001 non-null   str    
dtypes: float64(2), int64(5), str(1)
memory usage: 312.7 KB


In [6]:
df.describe()

,long_hair,forehead_width_cm,forehead_height_cm,nose_wide,nose_long,lips_thin,distance_nose_to_lip_long
count,5001.000000,5001.000000,5001.000000,5001.000000,5001.000000,5001.000000,5001.000000
mean,0.869626,13.181484,5.946311,0.493901,0.507898,0.493101,0.498900
std,0.336748,1.107128,0.541268,0.500013,0.499988,0.500002,0.500049
min,0.000000,11.400000,5.100000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,12.200000,5.500000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,13.100000,5.900000,0.000000,1.000000,0.000000,0.000000
75%,1.000000,14.000000,6.400000,1.000000,1.000000,1.000000,1.000000
max,1.000000,15.500000,7.100000,1.000000,1.000000,1.000000,1.000000


In [7]:
df.isnull().sum()

long_hair                    0
forehead_width_cm            0
forehead_height_cm           0
nose_wide                    0
nose_long                    0
lips_thin                    0
distance_nose_to_lip_long    0
gender                       0
dtype: int64

In [8]:
df.columns

Index(['long_hair', 'forehead_width_cm', 'forehead_height_cm', 'nose_wide',
       'nose_long', 'lips_thin', 'distance_nose_to_lip_long', 'gender'],
      dtype='str')

In [9]:
X = df.drop(columns=["gender"])
y = (df["gender"] == "Male").astype(int)

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

In [11]:
# -------------------------------------------------------------------------------
# Logistic Regression

from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=100, random_state=42)

lr.fit(X_train, y_train)

print(f"Accuracy Logistic Regression -> {round(lr.score(X_test, y_test), 4)}")

Accuracy Logistic Regression -> 0.971


In [12]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold

lr_pipeline = Pipeline([("sc", StandardScaler()), ("model", LogisticRegression())])

lr_param = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__solver": ["lbfgs", "liblinear"],
    "model__max_iter": [100, 200, 500],
    "model__class_weight": [None, "balanced"],
}

cv = StratifiedKFold(5, shuffle=True, random_state=42)

lr_gs = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=lr_param,
    cv=cv,
    scoring="f1_weighted",
    n_jobs=-1,
    verbose=1,
)

lr_gs.fit(X_train, y_train)

print("Best Parameters:", lr_gs.best_params_)
print("Best CV Score:", lr_gs.best_score_)
best_model = lr_gs.best_estimator_
y_pred = best_model.predict(X_test)

Fitting 5 folds for each of 60 candidates, totalling 300 fits
Best Parameters: {'model__C': 0.01, 'model__class_weight': None, 'model__max_iter': 100, 'model__solver': 'liblinear'}
Best CV Score: 0.9687488956453363


In [13]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(
    "Precision:", precision_score(y_test, y_pred, average="weighted", zero_division=0)
)
print("Recall:", recall_score(y_test, y_pred, average="weighted", zero_division=0))
print("F1 Score:", f1_score(y_test, y_pred, average="weighted", zero_division=0))
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.971028971028971
Precision: 0.9710309092105499
Recall: 0.971028971028971
F1 Score: 0.971028971028971
              precision    recall  f1-score   support

           0       0.97      0.97      0.97       501
           1       0.97      0.97      0.97       500

    accuracy                           0.97      1001
   macro avg       0.97      0.97      0.97      1001
weighted avg       0.97      0.97      0.97      1001



In [14]:
# -------------------------------------------------------------------------------
# Random Forest
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=1,
    criterion="gini",
    random_state=42,
    n_jobs=-1,
)

rf.fit(X_train, y_train)

print(f"Accuracy Random Forest -> {round(rf.score(X_test, y_test), 4)}")

Accuracy Random Forest -> 0.975


In [15]:
rf_param = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "criterion": ["gini", "entropy"],
}

rf_gs = GridSearchCV(
    estimator=rf,
    param_grid=rf_param,
    cv=5,
    scoring="f1_weighted",
    n_jobs=-1,
    verbose=1,
)

rf_gs.fit(X_train, y_train)

print("Best Parameters:", rf_gs.best_params_)
print("Best CV Score:", rf_gs.best_score_)
best_model = rf_gs.best_estimator_
y_pred = best_model.predict(X_test)

Fitting 5 folds for each of 216 candidates, totalling 1080 fits
Best Parameters: {'criterion': 'entropy', 'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
Best CV Score: 0.9724989803991686


In [16]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(
    "Precision:", precision_score(y_test, y_pred, average="weighted", zero_division=0)
)
print("Recall:", recall_score(y_test, y_pred, average="weighted", zero_division=0))
print("F1 Score:", f1_score(y_test, y_pred, average="weighted", zero_division=0))
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.9760239760239761
Precision: 0.976145246929664
Recall: 0.9760239760239761
F1 Score: 0.9760222530583438
              precision    recall  f1-score   support

           0       0.97      0.98      0.98       501
           1       0.98      0.97      0.98       500

    accuracy                           0.98      1001
   macro avg       0.98      0.98      0.98      1001
weighted avg       0.98      0.98      0.98      1001



In [17]:
# -------------------------------------------------------------------------------
# Decision Tree

from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=1,
    criterion="gini",
    random_state=42,
)

dt.fit(X_train, y_train)

print(f"Accuracy Decision Tree -> {round(dt.score(X_test, y_test), 4)}")

Accuracy Decision Tree -> 0.971


In [18]:
dt_param = {
    "max_depth": [None, 5, 10, 15, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "criterion": ["gini", "entropy", "log_loss"],
    "max_features": [None, "sqrt", "log2"],
}

dt_gs = GridSearchCV(
    estimator=dt, param_grid=dt_param, cv=5, scoring="f1_weighted", n_jobs=-1, verbose=1
)

dt_gs.fit(X_train, y_train)

print("Best Parameters:", dt_gs.best_params_)
print("Best CV Score:", dt_gs.best_score_)
best_model = dt_gs.best_estimator_
y_pred = best_model.predict(X_test)

Fitting 5 folds for each of 405 candidates, totalling 2025 fits
Best Parameters: {'criterion': 'gini', 'max_depth': 5, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 2}
Best CV Score: 0.9699951261505826


In [19]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(
    "Precision:", precision_score(y_test, y_pred, average="weighted", zero_division=0)
)
print("Recall:", recall_score(y_test, y_pred, average="weighted", zero_division=0))
print("F1 Score:", f1_score(y_test, y_pred, average="weighted", zero_division=0))
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.968031968031968
Precision: 0.9680988584756881
Recall: 0.968031968031968
F1 Score: 0.9680306279903865
              precision    recall  f1-score   support

           0       0.96      0.97      0.97       501
           1       0.97      0.96      0.97       500

    accuracy                           0.97      1001
   macro avg       0.97      0.97      0.97      1001
weighted avg       0.97      0.97      0.97      1001



In [20]:
# -------------------------------------------------------------------------------
# K Neighbors
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train, y_train)

print(f"Accuracy KNN -> {round(knn.score(X_test, y_test), 4)}")

Accuracy KNN -> 0.965


In [21]:
knn_pipeline = Pipeline([("sc", StandardScaler()), ("model", KNeighborsClassifier())])

knn_param = {
    "model__n_neighbors": [3, 5, 7, 9, 11, 15, 21],
    "model__weights": ["uniform", "distance"],
    "model__metric": ["euclidean", "manhattan"],
}

knn_gs = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=knn_param,
    cv=5,
    scoring="f1_weighted",
    n_jobs=-1,
    verbose=1,
)

knn_gs.fit(X_train, y_train)

print("Best Parameters:", knn_gs.best_params_)
print("Best CV Score:", knn_gs.best_score_)
best_model = knn_gs.best_estimator_
y_pred = best_model.predict(X_test)

Fitting 5 folds for each of 28 candidates, totalling 140 fits
Best Parameters: {'model__metric': 'euclidean', 'model__n_neighbors': 21, 'model__weights': 'uniform'}
Best CV Score: 0.9694978099138261


In [22]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(
    "Precision:", precision_score(y_test, y_pred, average="weighted", zero_division=0)
)
print("Recall:", recall_score(y_test, y_pred, average="weighted", zero_division=0))
print("F1 Score:", f1_score(y_test, y_pred, average="weighted", zero_division=0))
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.9690309690309691
Precision: 0.9691821297188495
Recall: 0.9690309690309691
F1 Score: 0.96902818710662
              precision    recall  f1-score   support

           0       0.96      0.98      0.97       501
           1       0.98      0.96      0.97       500

    accuracy                           0.97      1001
   macro avg       0.97      0.97      0.97      1001
weighted avg       0.97      0.97      0.97      1001



In [23]:
# -------------------------------------------------------------------------------
# SVM
from sklearn.svm import SVC

svm = SVC(C=1.0, kernel="rbf", gamma="scale", random_state=42)

svm.fit(X_train, y_train)

print(f"Accuracy SVM -> {round(svm.score(X_test, y_test), 4)}")

Accuracy SVM -> 0.97


In [24]:
svm_pipeline = Pipeline([("sc", StandardScaler()), ("model", SVC())])

svm_param = {
    "model__C": [0.1, 1, 10, 100],
    "model__kernel": ["linear", "rbf"],
    "model__gamma": ["scale", "auto"],
}

svm_gs = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=svm_param,
    cv=5,
    scoring="f1_weighted",
    n_jobs=-1,
    verbose=1,
)

svm_gs.fit(X_train, y_train)

print("Best Parameters:", svm_gs.best_params_)
print("Best CV Score:", svm_gs.best_score_)
best_model = svm_gs.best_estimator_
y_pred = best_model.predict(X_test)

Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best Parameters: {'model__C': 1, 'model__gamma': 'scale', 'model__kernel': 'rbf'}
Best CV Score: 0.9712480548434141


In [25]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(
    "Precision:", precision_score(y_test, y_pred, average="weighted", zero_division=0)
)
print("Recall:", recall_score(y_test, y_pred, average="weighted", zero_division=0))
print("F1 Score:", f1_score(y_test, y_pred, average="weighted", zero_division=0))
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.973026973026973
Precision: 0.9730438072993405
Recall: 0.973026973026973
F1 Score: 0.9730266499918739
              precision    recall  f1-score   support

           0       0.97      0.98      0.97       501
           1       0.98      0.97      0.97       500

    accuracy                           0.97      1001
   macro avg       0.97      0.97      0.97      1001
weighted avg       0.97      0.97      0.97      1001



In [ ]:
models = {
    "Logistic Regression": lr_gs,
    "Decision Tree": dt_gs,
    "Random Forest": rf_gs,
    "KNN": knn_gs,
    "SVM": svm_gs,
}

results = []

for name, grid in models.items():
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)

    results.append(
        {
            "Model": name,
            "CV F1": grid.best_score_,
            "Test Accuracy": accuracy_score(y_test, y_pred),
            "Test Precision": precision_score(
                y_test, y_pred, average="weighted", zero_division=0
            ),
            "Test Recall": recall_score(
                y_test, y_pred, average="weighted", zero_division=0
            ),
            "Test F1": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        }
    )

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(by="Test F1", ascending=False).reset_index(
    drop=True
)

results_df.round(4)

,Model,CV F1,Test Accuracy,Test Precision,Test Recall,Test F1
0,Random Forest,0.9725,0.976,0.9761,0.976,0.976
1,SVM,0.9712,0.973,0.9730,0.973,0.973
2,Logistic Regression,0.9687,0.971,0.9710,0.971,0.971
3,KNN,0.9695,0.969,0.9692,0.969,0.969
4,Decision Tree,0.9700,0.968,0.9681,0.968,0.968
